# GENAI – Modül 2 – Case Study 2: Fraud Detection

## Dengesiz Verilerde Karar Verme

Dijital ödeme / e-ticaret. Günde milyonlarca işlem, gerçek zamanlı. Amaç iki ucu birden tutmak: dolandırıcılık kaybı düşük kalsın, gerçek müşterinin ödemesi de boş yere kesilmesin.

Elinde ~**1.8 milyon** işlem var, bunların sadece **%1.4’ü** fraud. Problem baştan dengesiz. Üç şey denemişler:

| Yaklaşım | Rol | Precision | Recall |
|---|---|---:|---:|
| Logistic Regression | hızlı, düşük gecikmeli baseline | 0.81 | 0.22 |
| XGBoost | daha karmaşık örüntü | 0.74 | 0.61 |
| Zero-shot LLM sınıflandırıcı | etiket istemeden karar | 0.66 | 0.58 |

İş tarafında false negative (fraud’u kaçırmak) doğrudan para. False positive ise kuyruğu şişiriyor, inceleme kapasitesi de sınırlı. Bir de sert bir şey var: skor **250 ms** altında dönmezse ödeme deneyimi bozuluyor.

Soru “hangi model daha güzel yazdı?” değil. Hangi yaklaşım canlıya daha yakışır, hangi LLM bu kararı verirken JSON’dan dışarı taşmıyor.



# Görev 1 – Metriklerin Teknik Yorumu

## Soru

Fraud detection probleminde precision ve recall metrikleri neden kritik öneme sahiptir? Bu metriklerin iş maliyetiyle ilişkisini açıklayınız.


## Cevap

Accuracy’ye bakmak burada neredeyse kandırmaca. 1.8 milyon işlemin %1.4’ü fraud. Herkese “temiz” desen bile %98.6 doğru çıkarsın, şirket yine para kaybeder. O yüzden precision ve recall.

Recall, gerçek dolandırıcılığın ne kadarını yakaladığımız. Kaçırdığın her vaka false negative. Context de açık yazıyor: FN pahalı, çünkü işlem geçmiş, para gitmiş. Logistic Regression’ın precision’ı 0.81, kâğıtta güzel; recall’u 0.22. Alttaki hesaba göre ~25.200 fraud’un kabaca 19.600’ünü bırakıyor. Yani “fraud dedimse genelde haklıyım, ama çoğu fraud’u hiç demiyorum.”

Precision da “fraud” dediğimiz işlemlerin kaçı gerçekten fraud. Düşükse false positive şişer. Her FP bir işlemi durdurmak, müşteriyi kızdırmak, birinin kuyruğa bakması. İnceleme zaten kısıtlı. XGBoost 0.74 / 0.61; LR’ye göre daha çok yakalıyor, kuyruk da büyüyor. FN daha pahalı olduğu için bu takas bana daha mantıklı geliyor. LLM recall 0.58 ile XGBoost’a yakın, precision 0.66 ile daha çok yanlış alarm basıyor.

Kısaca recall kaçan para, precision operasyon yükü. İkisini birden okumadan canlıya alma kararı vermezdim. Bir de 250 ms var. Metriği iyi olup gecikmeyi tutmayan şey, ödeme hattında işe yaramaz.


Aynı JSON’dan kaba TP / FN / FP çıkardım. Yeni bir test değil, verilen precision–recall’u 1.8 milyona vurunca ne kadar işlem düşüyor, onu görmek için.


In [6]:
n = 1_800_000
fraud_rate = 0.014
n_fraud = n * fraud_rate

models = {
    "logistic_regression": {"precision": 0.81, "recall": 0.22},
    "xgboost": {"precision": 0.74, "recall": 0.61},
    "zero_shot_llm_classifier": {"precision": 0.66, "recall": 0.58},
}

print(f"Toplam işlem : {n:,}")
print(f"Fraud adedi  : {n_fraud:,.0f}  ({fraud_rate:.1%})")
print(f"Temiz işlem  : {n - n_fraud:,.0f}")
print()

for name, m in models.items():
    p, r = m["precision"], m["recall"]
    tp = r * n_fraud
    fn = n_fraud - tp
    fp = tp * (1 - p) / p
    f1 = 2 * p * r / (p + r)
    flags = tp + fp
    print(f"{name}")
    print(f"  precision={p:.2f}  recall={r:.2f}  F1={f1:.2f}")
    print(f"  yakalanan fraud (TP) : {tp:,.0f}")
    print(f"  kaçan fraud (FN)     : {fn:,.0f}")
    print(f"  yanlış alarm (FP)    : {fp:,.0f}")
    print(f"  inceleme kuyruğu     : {flags:,.0f}  (TP+FP)")
    print()



Toplam işlem : 1,800,000
Fraud adedi  : 25,200  (1.4%)
Temiz işlem  : 1,774,800

logistic_regression
  precision=0.81  recall=0.22  F1=0.35
  yakalanan fraud (TP) : 5,544
  kaçan fraud (FN)     : 19,656
  yanlış alarm (FP)    : 1,300
  inceleme kuyruğu     : 6,844  (TP+FP)

xgboost
  precision=0.74  recall=0.61  F1=0.67
  yakalanan fraud (TP) : 15,372
  kaçan fraud (FN)     : 9,828
  yanlış alarm (FP)    : 5,401
  inceleme kuyruğu     : 20,773  (TP+FP)

zero_shot_llm_classifier
  precision=0.66  recall=0.58  F1=0.62
  yakalanan fraud (TP) : 14,616
  kaçan fraud (FN)     : 10,584
  yanlış alarm (FP)    : 7,529
  inceleme kuyruğu     : 22,145  (TP+FP)



# Görev 2 – USER PROMPT TASARIMI

Aşağıdaki prompt üç modele de aynı gidecek. Production kararı isteyecek, metrikle iş kısıtını birlikte tartacak, JSON’da yoksa uydurtmayacak.


In [7]:
import json

CONTEXT_JSON = {
    "task": "fraud detection (binary classification)",
    "dataset_size": 1800000,
    "fraud_rate": 0.014,
    "models": ["logistic_regression", "xgboost", "zero_shot_llm_classifier"],
    "validation": {
        "logistic_regression": {"precision": 0.81, "recall": 0.22},
        "xgboost": {"precision": 0.74, "recall": 0.61},
        "zero_shot_llm_classifier": {"precision": 0.66, "recall": 0.58},
    },
    "constraints": {
        "false_negatives_costly": True,
        "manual_review_capacity_limited": True,
        "latency_ms_max": 250,
    },
}

USER_PROMPT = f"""Sen bir production karar destek asistanısın. Aşağıdaki CONTEXT_JSON bir fraud detection (ikili sınıflandırma) senaryosudur.

Görevin: üretim ortamı için en uygun yaklaşımı seçmek ve gerekçelendirmek.

Zorunlu kurallar:
1. Yalnızca CONTEXT_JSON içindeki bilgiyi kullan.
2. JSON'da olmayan metrik, maliyet tutarı, gecikme ölçümü, altyapı kapasitesi veya model yeteneği UYDURMA.
3. Bir bilgi yoksa açıkça "context'te yok" de. Tahmin etme.
4. Kararda şunları birlikte tart:
   - false negative maliyeti yüksek
   - manuel inceleme kapasitesi sınırlı
   - latency üst sınırı 250 ms
   - sınıf dengesizliği (fraud_rate = 0.014)
5. "Hangi metin daha akıcı?" diye değerlendirme. Hangi yaklaşım production'da daha güvenilir skorlama / karar destek aracı, ona bak.
6. Ana tavsiye tek olsun. Başka bir yaklaşımı ancak yardımcı rol olarak ve context'teki kısıtlarla çelişmeyecek şekilde önerebilirsin.

Çıktıyı TAM OLARAK aşağıdaki formatta ver. Önce veya sonra ek cümle yazma.

Model:
(kendi model adın: GPT / Gemini / Cohere)

Recommendation:
(seçilen yaklaşım)

Reasoning:
- Madde 1
- Madde 2

Risks:
- Risk 1
- Risk 2

Next actions:
- Aksiyon 1
- Aksiyon 2

CONTEXT_JSON:
{json.dumps(CONTEXT_JSON, indent=2)}
"""

print(USER_PROMPT)



Sen bir production karar destek asistanısın. Aşağıdaki CONTEXT_JSON bir fraud detection (ikili sınıflandırma) senaryosudur.

Görevin: üretim ortamı için en uygun yaklaşımı seçmek ve gerekçelendirmek.

Zorunlu kurallar:
1. Yalnızca CONTEXT_JSON içindeki bilgiyi kullan.
2. JSON'da olmayan metrik, maliyet tutarı, gecikme ölçümü, altyapı kapasitesi veya model yeteneği UYDURMA.
3. Bir bilgi yoksa açıkça "context'te yok" de. Tahmin etme.
4. Kararda şunları birlikte tart:
   - false negative maliyeti yüksek
   - manuel inceleme kapasitesi sınırlı
   - latency üst sınırı 250 ms
   - sınıf dengesizliği (fraud_rate = 0.014)
5. "Hangi metin daha akıcı?" diye değerlendirme. Hangi yaklaşım production'da daha güvenilir skorlama / karar destek aracı, ona bak.
6. Ana tavsiye tek olsun. Başka bir yaklaşımı ancak yardımcı rol olarak ve context'teki kısıtlarla çelişmeyecek şekilde önerebilirsin.

Çıktıyı TAM OLARAK aşağıdaki formatta ver. Önce veya sonra ek cümle yazma.

Model:
(kendi model adın: GPT / G

## Prompt neden böyle?

“Şunu değerlendir” deyince model boşlukları kendi dolduruyor. Maliyet uyduruyor, XGBoost 250 ms’i tutuyormuş gibi yazıyor. Aşağıda 0.7’de tam bunu yaptılar zaten.

O yüzden prompt’a üç şeyi çaktım:

- Tek karar iste, “belki hibrit düşünelim” kaçmasın.
- FN pahalı, inceleme sınırlı, 250 ms. Sadece precision–recall tablosuna bakıp çıkmasın.
- Sayı JSON’da yoksa “yok” desin. Uydurmasın.

Metin üç modelde de aynı. Fark onlarda kalsın, bende değil.


# Görev 3 – Temperature Deneyi

Aynı context, aynı prompt. Her model için bir kez **0.0**, bir kez **0.7**.

Key’leri notebook’a yazma. Bu klasördeki `.env` zaten okunuyor:

```text
OPENAI_API_KEY=...
GEMINI_API_KEY=...
COHERE_API_KEY=...
```


In [8]:
import os
from pathlib import Path

import requests


def load_dotenv(path: Path) -> None:
    if not path.exists():
        return
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


# notebook bu klasörde, .env genelde repo kökünde
here = Path.cwd()
for candidate in [here / ".env", here.parent / ".env", here.parent.parent / ".env"]:
    load_dotenv(candidate)

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY", "")
COHERE_API_KEY = os.environ.get("COHERE_API_KEY") or os.environ.get("CO_API_KEY", "")

MODELS = {
    "GPT": "gpt-4o-mini",
    "Gemini": "gemini-2.5-flash",
    "Cohere": "command-r-plus-08-2024",
}


def call_gpt(prompt: str, temperature: float) -> str:
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY yok")
    r = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}"},
        json={
            "model": MODELS["GPT"],
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=90,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def call_gemini(prompt: str, temperature: float) -> str:
    if not GEMINI_API_KEY:
        raise RuntimeError("GEMINI_API_KEY / GOOGLE_API_KEY yok")
    url = (
        "https://generativelanguage.googleapis.com/v1beta/models/"
        f"{MODELS['Gemini']}:generateContent"
    )
    r = requests.post(
        url,
        params={"key": GEMINI_API_KEY},
        json={
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {
                "temperature": temperature,
                "maxOutputTokens": 2048,
                "thinkingConfig": {"thinkingBudget": 0},
            },
        },
        timeout=90,
    )
    r.raise_for_status()
    parts = r.json()["candidates"][0]["content"]["parts"]
    return "".join(p.get("text", "") for p in parts)


def call_cohere(prompt: str, temperature: float) -> str:
    if not COHERE_API_KEY:
        raise RuntimeError("COHERE_API_KEY yok")
    r = requests.post(
        "https://api.cohere.com/v2/chat",
        headers={
            "Authorization": f"Bearer {COHERE_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": MODELS["Cohere"],
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=90,
    )
    r.raise_for_status()
    data = r.json()
    message = data.get("message", {})
    chunks = message.get("content", [])
    texts = [c.get("text", "") for c in chunks if c.get("type") == "text" or "text" in c]
    if texts:
        return "\n".join(texts)
    return data.get("text") or json.dumps(data)


CALLERS = {"GPT": call_gpt, "Gemini": call_gemini, "Cohere": call_cohere}

print("Anahtar durumu")
print(f"  GPT    : {'var' if OPENAI_API_KEY else 'YOK'}")
print(f"  Gemini : {'var' if GEMINI_API_KEY else 'YOK'}")
print(f"  Cohere : {'var' if COHERE_API_KEY else 'YOK'}")



Anahtar durumu
  GPT    : var
  Gemini : var
  Cohere : var


In [11]:
temps = [0.0, 0.7]
outputs = {}  # (model, temp) -> text

for model_name, fn in CALLERS.items():
    for temp in temps:
        key = (model_name, temp)
        print("=" * 72)
        print(f"{model_name} | temperature={temp}")
        print("=" * 72)
        try:
            text = fn(USER_PROMPT, temp)
            outputs[key] = text
            print(text)
        except Exception as exc:
            outputs[key] = f"[HATA] {exc}"
            print(outputs[key])
        print()



GPT | temperature=0.0
Model:
XGBoost

Recommendation:
XGBoost modelini kullanmak.

Reasoning:
- XGBoost, en yüksek recall değerine (0.61) sahip olduğu için, dolayısıyla daha fazla dolandırıcılığı tespit etme potansiyeline sahiptir. Bu, false negative maliyetinin yüksek olduğu bir senaryoda kritik öneme sahiptir.
- XGBoost'un precision değeri (0.74) makul bir seviyededir ve sınıf dengesizliği göz önüne alındığında, bu modelin performansı kabul edilebilir bir düzeydedir.

Risks:
- XGBoost'un precision değeri, manuel inceleme kapasitesinin sınırlı olduğu durumlarda yanlış pozitiflerin sayısını artırabilir.
- Modelin karmaşıklığı, bazı durumlarda daha uzun eğitim sürelerine yol açabilir, ancak bu durum context'te belirtilen latency kısıtlarıyla çelişmemektedir.

Next actions:
- XGBoost modelinin üretim ortamında test edilmesi ve performansının izlenmesi.
- Modelin sonuçlarına göre manuel inceleme süreçlerinin optimize edilmesi.

GPT | temperature=0.7
Model:
GPT

Recommendation:
xgboost

Re

## Temperature değişimi ne yaptı?

Üçü de XGBoost dedi, 0.0’da da 0.7’de de. Karar değişmedi. Değişen şey, cümlenin duruşu.

0.0 daha tutanak gibi. GPT “XGBoost modelini kullanmak” diye bitiriyor, neredeyse emir. Gemini JSON’daki alan adlarını (`false_negatives_costly`) tek tek sayıyor; biraz robot ama context’ten çıkmıyor. Cohere uzatıyor, yine XGBoost. Format da daha düzgün duruyor.

0.7’de kafa dağılıyor. GPT recommendation’ı birden `xgboost` yazmış, bir de “250 ms’i aşmadan çalışır” demiş. Context’te XGBoost’un kaç milisaniyede döndüğü yok. Prompt’ta uydurma demiştik, 0.7 orayı dinlemiyor.

Cohere daha kurnaz. “Gecikme belirtilmemiş” diyor, tam tamam diyorsun, cümlenin devamında “ama muhtemelen 250 ms’e sığar.” Eksiği görüp yine dolduruyor.

Gemini 0.7 risk satırına “latency ölçümü yok” yazmış, o kısım dürüst. 0.0’da ise “genel bilgiye göre XGBoost hızlıdır” diye yine dışarıdan bir şey sokmuş. En azından yalanı yalan diye işaretliyor.

Riskler de kayıyor. 0.0’da konu FN ve inceleme kuyruğu. 0.7’de GPT 0.74 precision’ı birden “düşük” ilan ediyor, Gemini müşteri deneyimine kayıyor, Cohere hiperparametre anlatmaya başlıyor. Aynı tablo, başka hikâye.

Bence production notu için 0.0 (ya da alttaki 0.2) daha sağlam. 0.7’de cümle güzelleşiyor, karar bozulmasa da gerekçe şişiyor.



# Görev 4 – Nihai Production Kararı

0.7’de spekülasyon kaçıyordu, o yüzden final’i **0.2**’de aldım. Referans aralık 0.0–0.2, karar notu gibi. Format case’teki şablon; her model ayrı.


In [12]:
FINAL_TEMP = 0.2
final_outputs = {}

for model_name, fn in CALLERS.items():
    print("=" * 72)
    print(f"{model_name} | temperature={FINAL_TEMP}")
    print("=" * 72)
    try:
        text = fn(USER_PROMPT, FINAL_TEMP)
        final_outputs[model_name] = text
        print(text)
    except Exception as exc:
        final_outputs[model_name] = f"[HATA] {exc}"
        print(final_outputs[model_name])
    print()



GPT | temperature=0.2
Model:
GPT

Recommendation:
XGBoost

Reasoning:
- XGBoost, en yüksek recall değerine sahip (0.61), bu da daha fazla dolandırıcılığı tespit etme potansiyeli sunar ve false negative maliyetinin yüksek olduğu bir senaryoda kritik öneme sahiptir.
- XGBoost'un precision değeri (0.74) kabul edilebilir bir seviyededir ve sınıf dengesizliği göz önüne alındığında, dolandırıcılık tespitinde daha dengeli bir performans sağlar.

Risks:
- XGBoost'un precision değeri, manuel inceleme kapasitesinin sınırlı olduğu durumlarda daha fazla yanlış pozitif sonuç üretebilir.
- Modelin karmaşıklığı, üretim ortamında beklenmedik gecikmelere neden olabilir, ancak 250 ms'lik üst sınır içinde kalması beklenmektedir.

Next actions:
- XGBoost modelinin üretim ortamında test edilmesi ve performansının izlenmesi.
- Yanlış pozitif oranını azaltmak için modelin ayarlanması ve gerektiğinde ek veri ile desteklenmesi.

Gemini | temperature=0.2
Model:
Gemini

Recommendation:
XGBoost

Reasoning:
- Fals

## Benim production kararım (temperature 0.2 bandı)

Modellerin yazdığına bakmadan, sadece JSON’a bakınca da aynı kapıya çıkıyorum. Yine de şablonu doldurayım, case öyle istiyor.

**Model:** öğrenci (GPT / Gemini / Cohere ile karşılaştırmak için)

**Recommendation:**  
Canlı skor için XGBoost. Zero-shot LLM ödeme hattına girmez. Logistic Regression tek başına fraud modeli olmaz.

**Reasoning:**
- FN burada para. LR recall 0.22 ile ~19.600 fraud bırakıyor. XGBoost 0.61, kaçanı kabaca yarıya indiriyor. Precision 0.81’den 0.74’e düşüyor, evet, ama kaçan paraya göre bu takas daha mantıklı.
- LLM recall 0.58, XGBoost’a yakın; precision 0.66, kuyruk daha şişkin. Bir de 250 ms var. Context’te LLM’in kaç ms döndüğü yok. Yoksa “tutar” diyemem, o yüzden canlı sınıfa koymam.
- LR kuyruğu küçük tutuyor çünkü neredeyse kimseyi fraud demiyor. İnceleme rahatlar, dolandırıcı da rahatlar. İş o değil.

**Risks:**
- XGBoost hâlâ ~9.800 vakayı kaçırıyor. “Seçtik, bitti” değil. Threshold, kural, kuyruk duruyor.
- LLM’i analiste özet yazdırmak ayrı iş. Aynı modeli 250 ms’lik yola sokarsak hem gecikme hem uydurma biner.

**Next actions:**
- XGBoost’u latency testine al, 250 ms’i tutuyor mu bak, tutuyorsa threshold’u FN’ye göre kaydır.
- LLM’i canlı sınıflandırıcı diye deneme; context’te hız kanıtı yok.
- LR’yi baseline / yedek tut, ana kapı yapma.

## Hangisi daha güvenilir karar destek aracı?

Güzel yazana bakmıyorum. 0.2’deki metinlere bakıyorum: kim JSON’da olmayanı “var” diye yutturuyor.

Üçü de XGBoost seçti. Hiçbiri LLM’i canlıya önermedi. O kısım tamam. Kaçak 250 ms’te.

Gemini en temiz duruyor. Kısıtları JSON’daki adlarıyla sayıyor, “ölçüm yok” diyor, next action’ı “git ölç” diye bitiriyor. Hâlâ bir yerde “genel bilgi, XGBoost hızlıdır” kaçırıyor ama en azından “context’te yok” diye kapıya yazıyor.

GPT daha kısa, karar net. Ama 0.2’de “250 ms içinde kalması beklenmektedir” diye ölçülmemiş şeyi onaylıyor. 0.0’da `Model:` satırına XGBoost yazmıştı, kendi adını unuttu. Küçük şey, yine de formatı bozuyor.

Cohere’i okurken en çok orada takıldım. Önce “context’te yok” diyor, cümlenin ikinci yarısında “varsayılabilir.” Recall 0.61’i bir yerde yüksek tutuyor, risks’te “hâlâ düşük.” Aynı sayı, iki hikâye.

Yani bu koşuda Gemini. En parlak metin o değil, en az yalan ekleyen o. GPT ikinci. Cohere aynı kapıya geliyor ama boşluğu en rahat dolduruyor.

Prompt uydurmayı yasaklamıştık. Temperature 0.7’de üçü de 250 ms’e kayıyor, 0.2’de bile tam durmuyorlar. O yüzden final’i düşük sıcaklıkta tutmak doğruymuş.
